In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [4]:
# Target
y = train["SalePrice"]

# Log transform target
y_log = np.log(y)

# Features
X = train.drop("SalePrice", axis=1)

In [5]:
# One-hot encoding
X_encoded = pd.get_dummies(X)

# Fill missing values
X_encoded = X_encoded.fillna(0)

print("Encoded shape:", X_encoded.shape)

Encoded shape: (1460, 288)


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y_log, test_size=0.2, random_state=42
)

In [7]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, random_state=42)

In [8]:
# Predict log values
val_preds_log = rf_model.predict(X_val)
train_preds_log = rf_model.predict(X_train)

# Convert back to real price
val_preds = np.exp(val_preds_log)
train_preds = np.exp(train_preds_log)

# Convert actual values back
y_val_actual = np.exp(y_val)
y_train_actual = np.exp(y_train)

In [9]:
# MAE
val_mae = mean_absolute_error(y_val_actual, val_preds)
train_mae = mean_absolute_error(y_train_actual, train_preds)

# RMSE
val_rmse = np.sqrt(mean_squared_error(y_val_actual, val_preds))
train_rmse = np.sqrt(mean_squared_error(y_train_actual, train_preds))

# R² Score
val_r2 = r2_score(y_val_actual, val_preds)
train_r2 = r2_score(y_train_actual, train_preds)

print("=== Training Metrics ===")
print("MAE:", train_mae)
print("RMSE:", train_rmse)
print("R2:", train_r2)

print("\n=== Validation Metrics ===")
print("MAE:", val_mae)
print("RMSE:", val_rmse)
print("R2:", val_r2)

=== Training Metrics ===
MAE: 7917.119319735047
RMSE: 12392.911433555037
R2: 0.9742504893041328

=== Validation Metrics ===
MAE: 17507.84701047156
RMSE: 29178.83439820946
R2: 0.8890002388633272


In [10]:
rf_model_cv = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

cv_scores = cross_val_score(
    rf_model_cv,
    X_encoded,
    y_log,
    cv=5,
    scoring="neg_mean_absolute_error"
)

cv_mae = -cv_scores

print("CV MAE (log scale):", cv_mae)
print("Average CV MAE:", cv_mae.mean())

CV MAE (log scale): [0.09666091 0.09871733 0.09598294 0.09311956 0.10037451]
Average CV MAE: 0.09697105105246813


In [11]:
# Train on full dataset
rf_model.fit(X_encoded, y_log)

importances = rf_model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

feature_importance.head(15)

,Feature,Importance
4,OverallQual,0.553058
16,GrLivArea,0.115590
12,TotalBsmtSF,0.046491
26,GarageCars,0.041259
27,GarageArea,0.022475
13,1stFlrSF,0.022220
9,BsmtFinSF1,0.021047
6,YearBuilt,0.015460
5,OverallCond,0.012405
3,LotArea,0.011500


In [12]:
# Encode test data
test_encoded = pd.get_dummies(test)

# Align columns
test_encoded = test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# Predict
test_preds_log = rf_model.predict(test_encoded)
test_preds = np.exp(test_preds_log)

# Create submission
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_preds
})

submission.to_csv("submission.csv", index=False)

print("Submission file created.")
submission.head()

Submission file created.


,Id,SalePrice
0,1461,124853.223062
1,1462,152967.193147
2,1463,176976.547637
3,1464,182155.940114
4,1465,198387.751179


Project Summary

This project developed an end-to-end regression pipeline to predict house prices using structured housing data. After preprocessing (missing value handling and one-hot encoding), a log transformation was applied to the target variable to address skewness and improve model stability. Multiple models were evaluated, with a tuned Random Forest outperforming the baseline Linear Regression model.

Model performance was assessed using MAE, RMSE, and R², along with 5-fold cross-validation to ensure robust generalization and reduce reliance on a single data split. Feature importance analysis revealed that overall material quality, living area, and basement size were the strongest predictors of price, aligning with domain expectations.

The final model was retrained on the full dataset and used to generate prediction outputs, demonstrating a complete machine learning workflow from data preprocessing to deployable results.